# TD Sémiologie - M2 IGAST - CORRECTION

## Une carte de la forêt

Surface des forêts (stock) en cercles proportionnels, puis part des feuillus (taux, en % de la surface forestière) en couleur dans les cercles. Discrétisation de Jenks en 5 classes.

In [1]:
import geopandas as gpd
import mappyng as mp

URL = "https://github.com/fbxyz/M2_IGAST/raw/master/data/{}_3857.geojson"

deps = gpd.read_file(URL.format("DEP_FORET")).to_crs(2154)
europe = gpd.read_file(URL.format("UE_GEN3")).to_crs(2154)
regions = deps.dissolve("insee_reg").reset_index()

# valeurs mises en forme pour les infobulles (virgule décimale)
deps["Forêt (ha)"] = deps["Surface foret"].map(lambda v: f"{v:,.0f}".replace(",", " "))
deps["Feuillus (%)"] = deps["p_feuillus"].map(lambda v: f"{v:.1f}".replace(".", ","))

deps[["LIB_GEO", "Surface foret", "p_feuillus"]].describe()

ERROR 1: PROJ: proj_create_from_database: Open of /home/florian/miniconda3/envs/geo/share/proj failed


,Surface foret,p_feuillus
count,96.000000,96.000000
mean,187002.691667,70.328125
std,113302.924208,20.468243
min,1057.100000,14.200000
25%,110768.325000,57.975000
50%,166841.700000,76.700000
75%,258826.450000,84.675000
max,599594.900000,98.600000


## Fond de carte

In [2]:
BBOX = [90000, 6040000, 1280000, 7150000]
SOURCE = "Source : IGN, 2024"


def fond():
    return (
        mp.Map(europe, bbox=BBOX, width=900, height=1000, padding=25,
               facecolor="#B9D9EB", border_radius=10,
               box_shadow={"dx": 3, "dy": 3, "blur": 6, "opacity": 0.2})
        + mp.OceanGradient(europe)
        + mp.Graticule(step=2, stroke="#ffffff", opacity=0.9, stroke_width=0.7)
        + mp.Basemap()
        + mp.Vector(deps, stroke="#9a9a9a", stroke_width=0.3)
        + mp.Vector(regions, stroke="#4d4d4d", stroke_width=0.4)
        + mp.DropShadow(europe, query="code_pays_ == 'FRA'")
        + mp.ScaleBar(length=100000, label="100 km", position=(0.55, 0.855), opacity=0.8)
        + mp.Source(SOURCE)
    )

## 1 - Surface des forêts

In [3]:
carte1 = (
    fond()
    + mp.Proportional(deps, column="Surface foret", fill="#2e7d32", fill_opacity=0.7,
                      stroke="#1b4d1e", legend=mp.Legend("Surface des forêts (ha)"))
    + mp.Title("Où se trouvent les forêts françaises ?",
               subtitle="Surface forestière par département, 2024")
    + mp.Tooltip(["LIB_GEO", "Forêt (ha)"], aliases={"LIB_GEO": "Département"})
)
carte1.to_interactive()

## 2 - Surface des forêts et part des feuillus

In [4]:
carte2 = (
    fond()
    + mp.Proportional(deps, column="Surface foret", fill="#d9d9d9", fill_opacity=1, stroke="#4d4d4d",
                      color_column="p_feuillus", cmap="Greens", method="FisherJenks", num_classes=5,
                      legend=mp.Legend("Surface des forêts (ha)", position={"x": 0.55, "y": 1.04}),
                      color_legend=mp.Legend("Part des feuillus dans la forêt (%)",
                                             decimal_sep=",", swatch_shape="rounded"))
    + mp.Title("Les feuillus dominent-ils partout ?",
               subtitle="Surface forestière et part des feuillus par département, 2024")
    + mp.Tooltip(["LIB_GEO", "Forêt (ha)", "Feuillus (%)"], aliases={"LIB_GEO": "Département"})
)
carte2.to_interactive()

In [5]:
carte1.render("foret_surface.html")
carte2.render("foret_surface_feuillus.html");